# Week 7, Day 4: PCA on neural populations: trajectories and dimensionality reduction

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2012)
N, C, T, K, dt = 60, 8, 50, 40, 0.02             # neurons, conditions, time bins, trials, bin width (s)
t = np.linspace(0, 1, T)
theta = np.arange(C) * 2 * np.pi / C
env = np.sin(np.pi * t)
z = np.zeros((3, C, T))
for c in range(C):
    phase = 2 * np.pi * 0.8 * t + theta[c]
    z[0, c], z[1, c], z[2, c] = env * np.cos(phase), env * np.sin(phase), t
mixing = rng.normal(size=(N, 3))
gain = rng.uniform(0.3, 3, N)                     # some neurons fire much more than others
rate = np.clip(gain[:, None, None] * (15 + 12 * np.einsum("nk,kct->nct", mixing, z)), 0, None)
counts = rng.poisson(rate[..., None] * dt, size=(N, C, T, K))
psth = counts.mean(axis=-1) / dt                  # trial-averaged rate in Hz, shape (60, 8, 50)
print(psth.shape)
fig, ax = plt.subplots(1, 3, figsize=(9, 2.5), sharex=True)
for a, n in zip(ax, [0, 1, 2]):
    a.plot(t, psth[n].T); a.set_title(f"neuron {n}"); a.set_xlabel("time (s)")
plt.tight_layout(); plt.show()

### Exercise: Build the data matrix

Write `to_matrix(R)` that turns an array of shape (N, C, T) into a (C·T) by N matrix whose row `c*T + t` is the population state for condition c at time t. Write `from_scores(S, C, T)` that reshapes a (C·T) by k matrix of scores into (C, T, k) for plotting.

In [ ]:
def to_matrix(R):
    n, c, t = R.shape
    return ...

def from_scores(S, C, T):
    return ...

print(to_matrix(psth).shape)

In [ ]:
import numpy as np
def test_shape():
    "to_matrix gives one row per condition and time bin, one column per neuron"
    assert to_matrix(psth).shape == (C * T, N)

def test_order():
    "row c*T + t holds the population state for condition c at time t"
    X = to_matrix(psth)
    assert np.allclose(X[3 * T + 7], psth[:, 3, 7])

def test_back():
    "from_scores turns a (C*T, k) score matrix back into (C, T, k)"
    S = np.arange(C * T * 2).reshape(C * T, 2)
    assert np.array_equal(from_scores(S, C, T)[3, 7], S[3 * T + 7])

test_shape()
test_order()
test_back()
print("All checks passed")

### Exercise: Soft-normalise and centre

Write `soft_norm(R)` for an (N, C, T) array, then build `X`, the centred (C·T) by N matrix of soft-normalised rates. Compare how much variance the three loudest neurons hold before and after.

In [ ]:
def soft_norm(R, c=5.0):
    span = ...                     # one value per neuron
    return ...

X = ...

raw = to_matrix(psth)
v_raw, v_norm = raw.var(axis=0), X.var(axis=0)
top3 = np.argsort(v_raw)[-3:]
print("share of variance in the 3 loudest neurons: raw", (v_raw[top3].sum() / v_raw.sum()).round(2), " normalised", (v_norm[top3].sum() / v_norm.sum()).round(2))

In [ ]:
import numpy as np
def test_soft():
    "soft_norm divides each neuron by its range over conditions and time plus 5 Hz"
    Rn = soft_norm(psth)
    rng_ = psth.max(axis=(1, 2)) - psth.min(axis=(1, 2))
    assert np.allclose(Rn, psth / (rng_ + 5)[:, None, None])

def test_ranges():
    "after soft normalisation no neuron has a range above 1"
    Rn = soft_norm(psth)
    assert np.all(Rn.max(axis=(1, 2)) - Rn.min(axis=(1, 2)) < 1)

def test_X():
    "X is the centred data matrix of the soft-normalised rates"
    assert X.shape == (C * T, N) and np.allclose(X.mean(axis=0), 0)
    assert np.allclose(X + to_matrix(soft_norm(psth)).mean(axis=0), to_matrix(soft_norm(psth)))

test_soft()
test_ranges()
test_X()
print("All checks passed")

### Exercise: PCA and trajectories

Take the SVD of `X`. Store the fraction of variance explained by every component in `var_explained`, and the scores on the first three components, reshaped to (C, T, 3), in `traj`. Plot the eight trajectories in the PC1 and PC2 plane, with a dot at the start of each, and a scree plot.

In [ ]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)
var_explained = ...
traj = ...

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for c in range(C):
    ax[0].plot(traj[c, :, 0], traj[c, :, 1], color=plt.cm.hsv(c / C)); ax[0].plot(traj[c, 0, 0], traj[c, 0, 1], "o", color=plt.cm.hsv(c / C))
ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2"); ax[0].set_aspect("equal")
ax[1].bar(np.arange(1, 13), var_explained[:12]); ax[1].set_xlabel("component"); ax[1].set_ylabel("variance explained")
plt.tight_layout(); plt.show()
print("top 3 explain", var_explained[:3].sum().round(3))

In [ ]:
import numpy as np
def test_var():
    "var_explained has one share per component, summing to 1, sorted"
    assert var_explained.shape == (N,) and np.isclose(var_explained.sum(), 1)
    assert np.all(np.diff(var_explained) <= 1e-12)

def test_top3():
    "the top three components explain more than 75% of the variance"
    assert var_explained[:3].sum() > 0.75

def test_traj():
    "traj holds the scores on the first three PCs, shaped (C, T, 3)"
    assert traj.shape == (C, T, 3)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    assert np.allclose(np.abs(traj[2, 10]), np.abs((X @ Vt[:3].T)[2 * T + 10]))

test_var()
test_top3()
test_traj()
print("All checks passed")

### Exercise: Participation ratio

Write `participation_ratio(lam)`. Compute it for the population (`pr_data`, from the squared singular values) and the number of components needed to reach 80% of the variance (`n80`). Which number would you report, and why do they differ?

In [ ]:
def participation_ratio(lam):
    return ...

pr_data = ...
n80 = ...
print("participation ratio", round(pr_data, 2), "  components for 80%:", n80)

In [ ]:
import numpy as np
def test_simple():
    "PR is 1 for one component and d for d equal components"
    assert np.isclose(participation_ratio(np.array([5.0, 0, 0])), 1)
    assert np.isclose(participation_ratio(np.ones(7)), 7)

def test_scale():
    "PR does not depend on the units of the eigenvalues"
    lam = np.array([4.0, 2.0, 1.0])
    assert np.isclose(participation_ratio(lam), participation_ratio(10 * lam))

def test_data():
    "pr_data is the PR of this dataset's eigenvalues, and n80 the components needed for 80%"
    assert np.isclose(pr_data, participation_ratio(s ** 2))
    assert n80 == int(np.searchsorted(np.cumsum(var_explained), 0.8) + 1)

test_simple()
test_scale()
test_data()
print("All checks passed")